# Batch Analysis with Atropos

Batch processing multiple scenarios and comparing results.

In [ ]:
import pandas as pd

from atropos import DeploymentScenario, estimate_outcome
from atropos.presets import STRATEGIES

## Define Multiple Scenarios

In [ ]:
scenarios = [
    DeploymentScenario(
        name="small-coder",
        parameters_b=7.0,
        memory_gb=5.0,
        throughput_toks_per_sec=25.0,
        power_watts=150.0,
        requests_per_day=20000,
        tokens_per_request=800,
        electricity_cost_per_kwh=0.15,
        annual_hardware_cost_usd=6000.0,
        one_time_project_cost_usd=8000.0,
    ),
    DeploymentScenario(
        name="medium-coder",
        parameters_b=34.0,
        memory_gb=14.0,
        throughput_toks_per_sec=40.0,
        power_watts=320.0,
        requests_per_day=50000,
        tokens_per_request=1200,
        electricity_cost_per_kwh=0.15,
        annual_hardware_cost_usd=24000.0,
        one_time_project_cost_usd=27000.0,
    ),
]

print(f"Defined {len(scenarios)} scenarios")

## Batch Evaluation

In [ ]:
results = []

for scenario in scenarios:
    for strategy_name, strategy in STRATEGIES.items():
        outcome = estimate_outcome(scenario, strategy)
        be_months = None
        if outcome.break_even_years:
            be_months = outcome.break_even_years * 12
        results.append({
            'Scenario': scenario.name,
            'Strategy': strategy_name,
            'Annual Savings ($)': outcome.annual_total_savings_usd,
            'Break-even (mo)': be_months,
            'CO2e Savings (kg/yr)': outcome.annual_co2e_savings_kg,
        })

df = pd.DataFrame(results)
df

## Analysis

In [ ]:
# Find best strategy per scenario
best = df.loc[df.groupby('Scenario')['Annual Savings ($)'].idxmax()]
best[['Scenario', 'Strategy', 'Annual Savings ($)']]

In [ ]:
# Summary statistics
summary = df.groupby('Scenario').agg({
    'Annual Savings ($)': ['min', 'max', 'mean'],
})
summary